# Perturbation theory: small pushes, order by order

**Professor Nano Labs · Quantum foundations**

**Try it live in your browser:** [The perturbation theory lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/perturbation-theory/)
**Watch first:** [Perturbation theory in quantum mechanics (non-degenerate) + Example](https://www.youtube.com/watch?v=Iodqi6rWa0E)
**Watch later:** [Time independent Schrodinger equation and stationary states](https://www.youtube.com/watch?v=cm8tf6NuePI) · [How an electron binds two protons, the hydrogen molecule ion](https://www.youtube.com/watch?v=4i9YHKyNS9g) · [How energy bands are formed? Key quantum concepts simplified](https://www.youtube.com/watch?v=Hm-v4Eby0hk)

Write $H=H^{(0)}+\lambda V$, where $H^{(0)}$ is solved, and expand the energy and the state in powers of $\lambda$, with intermediate normalisation $\langle n^{(0)}|n\rangle=1$. With $V_{mn}=\langle m^{(0)}|V|n^{(0)}\rangle$:
$$E_n^{(1)}=V_{nn},\qquad E_n^{(2)}=\sum_{m\ne n}\frac{|V_{mn}|^2}{E_n^{(0)}-E_m^{(0)}},\qquad |n^{(1)}\rangle=\sum_{m\ne n}\frac{V_{mn}}{E_n^{(0)}-E_m^{(0)}}|m^{(0)}\rangle,$$
$$|n^{(2)}\rangle=\sum_{m\ne n}\Big[\sum_{k\ne n}\frac{V_{mk}V_{kn}}{(E_n^{(0)}-E_m^{(0)})(E_n^{(0)}-E_k^{(0)})}-\frac{V_{nn}V_{mn}}{(E_n^{(0)}-E_m^{(0)})^2}\Big]|m^{(0)}\rangle .$$

The lab's system is a particle in a box, $0<x<L$, with lengths in $L$ and energies in $E_1=\pi^2\hbar^2/2mL^2$, so $E_n^{(0)}=n^2$. The push $V=s\,W(x)$ is a **tilt** $W=x-\tfrac12$, a **bump** $W=1$ for $|x-\tfrac12|<0.05$, or a **step** $W=1$ for $x>\tfrac12$. The notebook checks every number quoted in the lab's "Predict first" answers. It needs NumPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PI = np.pi
N = 40            # box states used, as in the lab
BUMP = 0.1        # width of the bump, in units of L

# ---------- matrix elements W_mn = <m|W|n> with psi_n = sqrt(2) sin(n pi x) ----------
def interval(m, n, a, b):
    # integral over [a, b] of 2 sin(m pi x) sin(n pi x)
    if m == n:
        return (b - a) - (np.sin(2 * n * PI * b) - np.sin(2 * n * PI * a)) / (2 * n * PI)
    d, s = m - n, m + n
    return (np.sin(d * PI * b) - np.sin(d * PI * a)) / (d * PI) - (np.sin(s * PI * b) - np.sin(s * PI * a)) / (s * PI)

def x_mn(m, n):
    # <m|x - 1/2|n>: -8mn / (pi^2 (m^2 - n^2)^2) when m + n is odd, zero otherwise
    return 0.0 if (m == n or (m + n) % 2 == 0) else -8 * m * n / (PI**2 * (m * m - n * n)**2)

KINDS = {
    'tilt': (lambda x: x - 0.5, x_mn),
    'bump': (lambda x: (np.abs(x - 0.5) < BUMP / 2).astype(float), lambda m, n: interval(m, n, 0.5 - BUMP / 2, 0.5 + BUMP / 2)),
    'step': (lambda x: (x > 0.5).astype(float), lambda m, n: interval(m, n, 0.5, 1.0)),
}
W = {k: np.array([[el(m, n) for n in range(1, N + 1)] for m in range(1, N + 1)]) for k, (_, el) in KINDS.items()}
E0 = np.arange(1, N + 1)**2.0

# check the formulas against direct numerical integration (midpoint rule, 400 000 points)
xs = (np.arange(400000) + 0.5) / 400000
worst = 0
for k, (Wx, _) in KINDS.items():
    for m, n in [(1, 1), (1, 2), (2, 3), (3, 1), (5, 8), (4, 4)]:
        num = np.mean(2 * np.sin(m * PI * xs) * np.sin(n * PI * xs) * Wx(xs))
        worst = max(worst, abs(num - W[k][m - 1, n - 1]))
print('largest difference from numerical integration:', worst)
assert worst < 1e-10

## Perturbation theory and the exact answer

`series` returns the first- and second-order energies and the first- and second-order coefficients $c_m$, with $c_n=0$ by intermediate normalisation. `exact` diagonalises $E_n^{(0)}\delta_{mn}+sW_{mn}$ in the same 40 states; in one dimension levels never cross, so level $n$ is the $n$-th lowest. As an independent check, `fd_levels` solves the box by finite differences on a grid of 4000 points, with no basis at all.

In [ ]:
def series(kind, n, s):
    V = s * W[kind]; i = n - 1; d = E0[i] - E0           # d[m] = E_n - E_m
    others = np.arange(N) != i
    c1 = np.zeros(N); c1[others] = V[others, i] / d[others]
    E1 = V[i, i]; E2 = np.sum(V[others, i]**2 / d[others])
    c2 = np.zeros(N)
    for m in np.where(others)[0]:
        c2[m] = np.sum(V[m, others] * V[others, i] / (d[m] * d[others])) - V[i, i] * V[m, i] / d[m]**2
    return dict(E1=E1, E2=E2, c1=c1, c2=c2, contrib=np.where(others, V[:, i]**2 / np.where(others, d, 1), 0))

def exact(kind, s):
    vals, vecs = np.linalg.eigh(np.diag(E0) + s * W[kind])
    vecs = vecs * np.sign(np.diag(vecs))                  # <k0|k> > 0
    return vals, vecs

def fd_levels(kind, s, M=4000, k=6):
    # -psi'' / pi^2 + s W psi = E psi on the open interval, with psi = 0 at the walls (energies in E1)
    h = 1 / (M + 1); x = np.arange(1, M + 1) * h
    main = 2 / (PI * h)**2 + s * KINDS[kind][0](x); off = -np.ones(M - 1) / (PI * h)**2
    return np.linalg.eigvalsh(np.diag(main) + np.diag(off, 1) + np.diag(off, -1))[:k]

for kind, s in [('tilt', 5), ('bump', 10), ('step', 2), ('tilt', 30)]:
    diff = np.max(np.abs(exact(kind, s)[0][:5] - fd_levels(kind, s)[:5]))
    print(f'{kind:5s} s = {s:2d}: 40-state basis against finite differences, largest difference {diff:.1e} E1')
    assert diff < 1e-3

# at weak pushes, perturbation theory and the exact answer differ only at third order and beyond
for kind in KINDS:
    errs = []
    for s in [0.2, 0.1, 0.05]:
        S = series(kind, 1, s); errs.append(abs(exact(kind, s)[0][0] - (1 + S['E1'] + S['E2'])))
    print(kind, 'error of second order at s = 0.2, 0.1, 0.05:', ['%.2e' % e for e in errs], ' ratio', round(errs[0] / errs[1], 1))
    assert errs[0] / errs[1] > 7.5        # at least s^3

## 1 · Energies, order by order (Questions 1 to 3)

In [ ]:
def energies(kind, n, s):
    S = series(kind, n, s); ex = exact(kind, s)[0][n - 1]
    return n * n, n * n + S['E1'], n * n + S['E1'] + S['E2'], ex

# Question 1: the tilted ground state
e0, e1, e2, ex = energies('tilt', 1, 5)
k = series('tilt', 1, 1)['E2']
closed = -64 / PI**4 * sum(m * m / (m * m - 1)**5 for m in range(2, 400, 2))
print(f'tilt s = 5, level 1: first order {e1:.3f}, second order {e2:.3f}, exact {ex:.3f};  E2 = {k:.5f} s^2 (closed form {closed:.5f})')
assert e1 == 1 and round(k, 5) == -0.01083 and abs(k - closed) < 1e-9
assert round(e2 - 1, 3) == -0.271 and round(e2, 3) == 0.729 and round(ex, 3) == 0.741
assert all(series('tilt', 1, 5)['contrib'] <= 0)      # no term can push the ground level up

# Question 2: a step of height 2
for n, (shift2, exn) in zip([1, 2], [(-0.25, 1.766), (0.19, None)]):
    e0, e1, e2, ex = energies('step', n, 2)
    print(f'step s = 2, level {n}: first order {e1:.4f}, second order {e2:.4f} (E2 = {e2 - e1:+.4f}), exact {ex:.4f}')
    assert abs((e1 - e0) - 1) < 1e-12 and abs((e2 - e1) - shift2) < 0.005
    if exn: assert round(ex, 3) == exn and round(e2, 2) == 1.75
shifts = [float(series('step', n, 2)['E1']) for n in range(1, 6)]
print('first-order shift of levels 1-5:', [round(v, 6) for v in shifts])
assert all(abs(v - 1) < 1e-12 for v in shifts)

# Question 3: level 2, tilted
S = series('tilt', 2, 5); e0, e1, e2, ex = energies('tilt', 2, 5)
print(f'tilt s = 5, level 2: push from level 1 {S["contrib"][0]:+.3f}, from level 3 {S["contrib"][2]:+.3f}; second order {e2 - 4:+.3f}, exact {ex - 4:+.3f}')
assert round(S['contrib'][0], 3) == 0.270 and round(S['contrib'][2], 3) == -0.189 and round(e2 - 4, 3) == 0.081 and round(ex - 4, 3) == 0.069

# the lab's physical scale: a 10 nm GaAs well
hbar, me, e = 1.054571817e-34, 9.1093837e-31, 1.602176634e-19
E1 = PI**2 * hbar**2 / (2 * 0.067 * me * (10e-9)**2) / e
print(f'E1 for a 10 nm GaAs well: {1000 * E1:.1f} meV; a tilt of s = 1 is {E1 / 10e-9 / 1e5:.0f} kV/cm')
assert round(1000 * E1) == 56 and round(E1 / 10e-9 / 1e5) == 56

In [ ]:
ss = np.linspace(0, 30, 121)
ex = [exact('tilt', s)[0][0] for s in ss]
p2 = [1 + series('tilt', 1, s)['E2'] for s in ss]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(ss, ex, label='exact'); ax[0].plot(ss, np.ones_like(ss), '--', label='first order'); ax[0].plot(ss, p2, label='second order')
ax[0].set_ylim(-5, 1.5); ax[0].set_xlabel('tilt strength s (E1)'); ax[0].set_ylabel('level 1 (E1)'); ax[0].legend()
c = series('tilt', 2, 5)['contrib'][:8]
ax[1].bar(np.arange(1, 9), c, color=['tab:red' if v > 0 else 'tab:blue' for v in c])
ax[1].set_xlabel('other level m'); ax[1].set_ylabel('push on level 2 (E1)'); ax[1].set_title('tilt s = 5: who pushes level 2')
plt.tight_layout(); plt.show()

## 2 · How the state changes (Questions 4 to 6)

With intermediate normalisation the corrected state is $|n^{(0)}\rangle+\sum_m c_m|m^{(0)}\rangle$, whose length is $\sqrt{1+\sum|c_m|^2}$. The exact eigenvector, scaled so that its $n$-th component is 1, can be compared with it coefficient by coefficient.

In [ ]:
def coeffs(kind, n, s, order):
    S = series(kind, n, s); c = np.zeros(N); c[n - 1] = 1
    if order >= 1: c += S['c1']
    if order >= 2: c += S['c2']
    return c

def exact_coeffs(kind, n, s):
    v = exact(kind, s)[1][:, n - 1]
    return v / v[n - 1]                                   # intermediate normalisation

overlap2 = lambda a, b: (a @ b)**2 / ((a @ a) * (b @ b))
trap = getattr(np, 'trapezoid', None) or np.trapz

# Question 4: what the tilted ground state borrows
c1, cx = coeffs('tilt', 1, 5, 1), exact_coeffs('tilt', 1, 5)
print('first order c_m, m = 1..6:', np.round(c1[:6], 4))
print('exact       c_m, m = 1..6:', np.round(cx[:6], 4))
assert np.all(c1[2::2] == 0)                              # odd states 3, 5, ... are not linked to state 1
assert round(c1[1], 3) == 0.300 and round(cx[1], 3) == 0.287 and round(c1[3], 3) == 0.005
# the odd states are not linked directly, but they come in at second order, through state 2
c2nd = coeffs('tilt', 1, 5, 2)
print(f'c_3: first order 0, second order {c2nd[2]:.3f}, exact {cx[2]:.3f}')
assert round(cx[2], 3) == 0.035 and round(c2nd[2], 3) == 0.037
psi = lambda c, x: sum(cm * np.sqrt(2) * np.sin((m + 1) * PI * x) for m, cm in enumerate(c))
x = np.linspace(0, 1, 2001); dens = psi(c1, x)**2 / trap(psi(c1, x)**2, x)
print(f'centre of the first-order state: <x> = {trap(x * dens, x):.3f} L (it moves to the low side of the tilt, x < 1/2)')
assert trap(x * dens, x) < 0.5

# Question 5: intermediate normalisation
S = series('tilt', 1, 5); length2 = 1 + np.sum(S['c1']**2)
share_exact = exact('tilt', 5)[1][0, 0]**2
print(f'length of the first-order state: {np.sqrt(length2):.3f}; share of psi_1 once normalised: {1 / length2:.3f} (exact {share_exact:.3f})')
assert round(np.sqrt(length2), 3) == 1.044 and round(length2, 3) == 1.090 and round(1 / length2, 3) == 0.917 and round(share_exact, 3) == 0.923

# Question 6: a strong tilt
c1, c2, cx = coeffs('tilt', 1, 20, 1), coeffs('tilt', 1, 20, 2), exact_coeffs('tilt', 1, 20)
e0, e1, e2, ex = energies('tilt', 1, 20)
print(f'tilt s = 20: c_2 first order {c1[1]:.2f}, exact {cx[1]:.2f}; overlap {overlap2(cx, c1):.3f} (first order), {overlap2(cx, c2):.3f} (second); energy {e2:.2f} against {ex:.2f}')
assert round(c1[1], 2) == 1.20 and round(cx[1], 2) == 0.82 and round(overlap2(cx, c1), 3) == 0.903 and round(overlap2(cx, c2), 3) == 0.950
print(f'exact c_3 at s = 20: {cx[2]:.2f}, which first order misses entirely (c_3 = {c1[2]:.0f})')
assert round(cx[2], 2) == 0.32 and c1[2] == 0
assert round(e2, 2) == -3.33 and round(ex, 2) == -1.97

# each order removes the next power of s from 1 - overlap: slopes 2, 4, 6 on a log-log plot
for order in range(3):
    a, b = [1 - overlap2(exact_coeffs('tilt', 1, s), coeffs('tilt', 1, s, order)) for s in (0.2, 0.1)]
    print(f'order {order}: 1 - overlap scales as s^{np.log2(a / b):.2f}')
    assert abs(np.log2(a / b) - 2 * (order + 1)) < 0.1

## 3 · Two levels repel (Questions 7 and 8)

The video's closing example, as an electron in two quantum dots: $|1\rangle$ on the left at $E_1=-\delta/2$, $|2\rangle$ on the right at $E_2=+\delta/2$, coupled by $V$, all in meV.

In [ ]:
def two(delta, V):
    vals, vecs = np.linalg.eigh(np.array([[-delta / 2, V], [V, delta / 2]]))
    return vals, vecs[0, 0]**2                            # levels, and the lower state's weight on |1> (left dot)

# Question 7: far apart
vals, w1 = two(1.0, 0.2)
shift, pt = vals[1] - 0.5, 0.2**2 / 1.0
keep = 1 / (1 + (0.2 / 1.0)**2)
print(f'delta = 1, V = 0.2: each level moves by {shift:.4f} meV (second order {pt:.3f}); lower state {100 * w1:.1f}% left (first order {100 * keep:.1f}%)')
assert round(pt, 3) == 0.040 and round(shift, 4) == 0.0385 and round(100 * w1, 1) == 96.4 and round(100 * keep, 1) == 96.2
assert vals[0] < -0.5 < 0.5 < vals[1]                     # level repulsion

# Question 8: on resonance
vals, w1 = two(0.0, 0.2)
print(f'delta = 0: levels {vals}, gap {vals[1] - vals[0]:.3f} meV, lower state {100 * w1:.0f}% left')
assert np.allclose(vals, [-0.2, 0.2]) and abs(w1 - 0.5) < 1e-12

# the gap never closes: the smallest gap over all detunings is 2V
ds = np.linspace(-2, 2, 4001)
gaps = [np.diff(two(d, 0.2)[0])[0] for d in ds]
print(f'smallest gap over -2 < delta < 2: {min(gaps):.4f} meV = 2V')
assert abs(min(gaps) - 0.4) < 1e-9

fig, ax = plt.subplots(figsize=(6, 3.6))
lev = np.array([two(d, 0.2)[0] for d in ds])
ax.plot(ds, -ds / 2, '--', color='grey'); ax.plot(ds, ds / 2, '--', color='grey')
ax.plot(ds, lev, color='tab:blue')
with np.errstate(divide='ignore', invalid='ignore'):
    ax.plot(ds, -ds / 2 - 0.04 / ds, ':', color='tab:orange'); ax.plot(ds, ds / 2 + 0.04 / ds, ':', color='tab:orange')
ax.set_ylim(-1.4, 1.4); ax.set_xlabel('detuning δ (meV)'); ax.set_ylabel('energy (meV)'); ax.set_title('avoided crossing, V = 0.2 meV')
plt.tight_layout(); plt.show()